# MLFLOW setup in Jupyter Notebook

This notebook contains a small Data Science Project where MLFlow is used to log all Feature Engineering and Modeling Parameters as well as Metrics.



## Loading Data

In [ ]:
import sys
# adding to the path variables the one folder higher (locally, not changing system variables)
sys.path.append("..")
import pandas as pd
import numpy as np
import warnings
import mlflow

from modeling.config import EXPERIMENT_NAME
TRACKING_URI = open("../.mlflow_uri").read().strip()

warnings.filterwarnings('ignore')

# coffee data
url="https://github.com/jldbc/coffee-quality-database/raw/master/data/robusta_data_cleaned.csv"
coffee_features=pd.read_csv(url)

# coffee score
url="https://raw.githubusercontent.com/jldbc/coffee-quality-database/master/data/robusta_ratings_raw.csv"
coffee_quality=pd.read_csv(url)

In [ ]:
coffee_features.head()

In [ ]:
coffee_quality.head()

In [ ]:
Y = coffee_quality["quality_score"]

## Data cleaning and feature engineering

In [ ]:
coffee_features.info()

In [ ]:
#for this exercise we will only deal with numeric variables

X = coffee_features.select_dtypes(['number'])

## Splitting data for testing 

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.20, random_state=42)

In [ ]:
#dropping Quakers column and unnamed
#changing one of the altitude to log and droping the original
X_train["altitude_mean_log"] = np.log(X_train["altitude_mean_meters"])
X_train.drop(['altitude_mean_meters'], axis=1, inplace=True)
X_train.drop(['Quakers'], axis=1, inplace=True)
X_train.drop(['Unnamed: 0'], axis=1, inplace=True)

In [ ]:
X_train.info()

In [ ]:
altitude_low_meters_mean = X_train["altitude_low_meters"].mean()
altitude_high_meters_mean = X_train["altitude_high_meters"].mean()
altitude_mean_log_mean = X_train["altitude_mean_log"].mean()

In [ ]:
# fillna with mean.. 
X_train["altitude_low_meters"] = X_train["altitude_low_meters"].fillna(altitude_low_meters_mean)
X_train["altitude_high_meters"] = X_train["altitude_high_meters"].fillna(altitude_high_meters_mean)
X_train["altitude_mean_log"] = X_train["altitude_mean_log"].fillna(altitude_mean_log_mean)

In [ ]:
print(f"altitude low meters mean is {altitude_low_meters_mean}")
print(f"altitude_high_meters_mean is {altitude_high_meters_mean}")
print(f"altitude_mean_log_mean is {altitude_mean_log_mean}")

## Trainining the model and tracking with MLFlow

In [ ]:
# setting the MLFlow connection and experiment
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.start_run()
run = mlflow.active_run()

In [ ]:
print("Active run_id: {}".format(run.info.run_id))

In [ ]:
#training the model
from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(X_train, y_train)

In [ ]:
#model metric on training data
from sklearn.metrics import mean_squared_error
y_train_pred = reg.predict(X_train)
mse_train = mean_squared_error(y_train, y_train_pred)
print(mse_train)

In [ ]:
# feature engineering of test data
#dropping Quakers column and unnamed
#changing one of the altitude to log and droping the original
X_test["altitude_mean_log"] = np.log(X_test["altitude_mean_meters"])
X_test.drop(['altitude_mean_meters'], axis=1, inplace=True)
X_test.drop(['Quakers'], axis=1, inplace=True)
X_test.drop(['Unnamed: 0'], axis=1, inplace=True)
# fillna with mean.. 
X_test["altitude_low_meters"] = X_test["altitude_low_meters"].fillna(altitude_low_meters_mean)
X_test["altitude_high_meters"] = X_test["altitude_high_meters"].fillna(altitude_high_meters_mean)
X_test["altitude_mean_log"] = X_test["altitude_mean_log"].fillna(altitude_mean_log_mean)

In [ ]:
#model metric on test data
y_test_pred = reg.predict(X_test)
mse_test = mean_squared_error(y_test, y_test_pred)
print(mse_test)

In [ ]:
#seting parameters that should be logged on MLFlow
#these parameters were used in feature engineering (inputing missing values)
#or parameters of the model (fit_intercept for Linear Regression model)
params = {
      "altitude_low_meters_mean": altitude_low_meters_mean,
      "altitude_high_meters_mean": altitude_high_meters_mean,
      "altitude_mean_log_mean": altitude_mean_log_mean,
      "fit_intercept": True,
  }

In [ ]:
#logging params to mlflow
mlflow.log_params(params)
#setting tags
mlflow.set_tag("running_from_jupyter", "True")
#logging metrics
mlflow.log_metric("train-" + "MSE", mse_train)
mlflow.log_metric("test-" + "MSE", mse_test)
# logging the model to mlflow will not work without a AWS Connection setup.. too complex for now
# but possible if running mlflow locally
# mlflow.log_artifact("../models")
# mlflow.sklearn.log_model(reg, "model")
mlflow.end_run()

In [ ]:
mlflow.get_run(run_id=run.info.run_id)

## Checking the experiments

while the next cell is running you will not be able to run other cells in the notebook

In [ ]:
!mlflow ui

In [ ]:
####

In [58]:
import duckdb
import pandas as pd
import openmeteo_requests
import requests_cache
from retry_requests import retry

consumption = pd.read_csv("../data/Actual_consumption_202303010000_202603020000_Hour.csv", delimiter=";")
prices = pd.read_csv("../data/Day-ahead_prices_202303010000_202603020000_Hour.csv", delimiter=";")
generation = pd.read_csv("../data/Forecasted_generation_Day-Ahead_202303010000_202603020000_Hour.csv", delimiter = ";")

In [79]:
prices.head(2)

,Start date,End date,Germany/Luxembourg [€/MWh] Calculated resolutions,∅ DE/LU neighbours [€/MWh] Calculated resolutions,Belgium [€/MWh] Calculated resolutions,Denmark 1 [€/MWh] Calculated resolutions,Denmark 2 [€/MWh] Calculated resolutions,France [€/MWh] Calculated resolutions,Netherlands [€/MWh] Calculated resolutions,Norway 2 [€/MWh] Calculated resolutions,Austria [€/MWh] Calculated resolutions,Poland [€/MWh] Calculated resolutions,Sweden 4 [€/MWh] Calculated resolutions,Switzerland [€/MWh] Calculated resolutions,Czech Republic [€/MWh] Calculated resolutions,DE/AT/LU [€/MWh] Calculated resolutions,Northern Italy [€/MWh] Calculated resolutions,Slovenia [€/MWh] Calculated resolutions,Hungary [€/MWh] Calculated resolutions
0,"Mar 1, 2023 12:00 AM","Mar 1, 2023 1:00 AM",133.08,125.46,123.80,133.08,133.08,145.0,125.23,112.47,135.99,134.13,60.40,142.28,134.60,-,145.00,136.18,135.26
1,"Mar 1, 2023 1:00 AM","Mar 1, 2023 2:00 AM",125.46,120.17,132.84,125.46,125.46,140.0,119.90,111.46,123.38,133.53,44.45,139.09,126.32,-,140.00,131.33,127.89


In [54]:
consumption.head(2)

,Start date,End date,grid load [MWh] Calculated resolutions,Grid load incl. hydro pumped storage [MWh] Calculated resolutions,Hydro pumped storage [MWh] Calculated resolutions,Residual load [MWh] Calculated resolutions
0,"Mar 1, 2023 12:00 AM","Mar 1, 2023 1:00 AM","48,971.75","49,576.50",604.75,"42,916.75"
1,"Mar 1, 2023 1:00 AM","Mar 1, 2023 2:00 AM","47,543.75","48,288.50",744.75,"41,641.25"


In [77]:
generation.head(2)

,Start date,End date,Total [MWh] Calculated resolutions,Photovoltaics and wind [MWh] Calculated resolutions,Wind offshore [MWh] Calculated resolutions,Wind onshore [MWh] Calculated resolutions,Photovoltaics [MWh] Calculated resolutions,Other [MWh] Calculated resolutions
0,"Mar 1, 2023 12:00 AM","Mar 1, 2023 1:00 AM","47,799.00","6,559.75",243.25,"6,316.50",0.00,"41,239.25"
1,"Mar 1, 2023 1:00 AM","Mar 1, 2023 2:00 AM","46,172.00","6,381.00",243.75,"6,137.25",0.00,"39,791.00"


In [55]:
consumption = consumption.rename(columns={"Start date": "timestamp"})
prices = prices.rename(columns={"Start date": "timestamp"})
generation = generation.rename(columns={"Start date": "timestamp"})

In [ ]:
df = duckdb.query("""
    SELECT 
        p.timestamp,
        p."Germany/Luxembourg [€/MWh] Calculated resolutions" as price,
        c."grid load [MWh] Calculated resolutions" as load,
        g."Wind offshore [MWh] Calculated resolutions" as wind_offshore,
        g."Wind onshore [MWh] Calculated resolutions" as wind_onshore,
        g."Photovoltaics [MWh] Calculated resolutions" as solar
    FROM prices p
    JOIN consumption c ON p.timestamp = c.timestamp
    JOIN generation g ON p.timestamp = g.timestamp
""").fetchdf()

print(df.head())
print(df.shape)
print(df.isnull().sum()) 

              timestamp   price       load wind_offshore wind_onshore solar
0  Mar 1, 2023 12:00 AM  133.08  48,971.75        243.25     6,316.50  0.00
1   Mar 1, 2023 1:00 AM  125.46  47,543.75        243.75     6,137.25  0.00
2   Mar 1, 2023 2:00 AM  124.96  47,152.00        229.00     5,835.25  0.00
3   Mar 1, 2023 3:00 AM  127.66  47,447.25        198.75     5,595.00  0.00
4   Mar 1, 2023 4:00 AM  130.58  49,152.25        154.00     5,600.00  0.00
(26346, 6)
timestamp        0
price            0
load             0
wind_offshore    0
wind_onshore     0
solar            0
dtype: int64


In [70]:
df.shape

(26346, 6)

In [ ]:
import openmeteo_requests
import requests_cache
from retry_requests import retry
import pandas as pd

cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

cities = {
    "Berlin":  (52.52, 13.41),
    "Munich":  (48.14, 11.58),
    "Hamburg": (53.55, 10.00),
    "Cologne": (50.94,  6.96)
}

temp_dfs = []  

for city, (lat, lon) in cities.items():
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2023-03-01",
        "end_date": "2026-03-01",
        "hourly": "temperature_2m",
        "timezone": "Europe/Berlin"
    }
    
    responses = openmeteo.weather_api(
        "https://archive-api.open-meteo.com/v1/archive", 
        params=params
    )
    response = responses[0]
    hourly = response.Hourly()
    
    temp_df = pd.DataFrame({
        "timestamp": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        f"temp_{city}": hourly.Variables(0).ValuesAsNumpy()
    })
    
    temp_dfs.append(temp_df)
    print(f"✅ {city} downloaded!")

temp_final = temp_dfs[0]
for t in temp_dfs[1:]:
    temp_final = temp_final.merge(t, on="timestamp")

temp_final["temperature"] = temp_final[
    ["temp_Berlin", "temp_Munich", "temp_Hamburg", "temp_Cologne"]
].mean(axis=1)

temp_final = temp_final[["timestamp", "temperature"]]

print(temp_final.head())
print(temp_final.shape)

✅ Berlin downloaded!
✅ Munich downloaded!
✅ Hamburg downloaded!
✅ Cologne downloaded!
                  timestamp  temperature
0 2023-02-28 23:00:00+00:00      -2.1500
1 2023-03-01 00:00:00+00:00      -2.8375
2 2023-03-01 01:00:00+00:00      -3.2750
3 2023-03-01 02:00:00+00:00      -3.7250
4 2023-03-01 03:00:00+00:00      -3.7125
(26328, 2)


In [ ]:
# checkingto match therows of both

# missing timestamps
smard_timestamps = set(df["timestamp"])
temp_timestamps = set(temp_final["timestamp"])

missing = smard_timestamps - temp_timestamps
print(sorted(missing))

['Apr 1, 2023 10:00 AM', 'Apr 1, 2023 10:00 PM', 'Apr 1, 2023 11:00 AM', 'Apr 1, 2023 11:00 PM', 'Apr 1, 2023 12:00 AM', 'Apr 1, 2023 12:00 PM', 'Apr 1, 2023 1:00 AM', 'Apr 1, 2023 1:00 PM', 'Apr 1, 2023 2:00 AM', 'Apr 1, 2023 2:00 PM', 'Apr 1, 2023 3:00 AM', 'Apr 1, 2023 3:00 PM', 'Apr 1, 2023 4:00 AM', 'Apr 1, 2023 4:00 PM', 'Apr 1, 2023 5:00 AM', 'Apr 1, 2023 5:00 PM', 'Apr 1, 2023 6:00 AM', 'Apr 1, 2023 6:00 PM', 'Apr 1, 2023 7:00 AM', 'Apr 1, 2023 7:00 PM', 'Apr 1, 2023 8:00 AM', 'Apr 1, 2023 8:00 PM', 'Apr 1, 2023 9:00 AM', 'Apr 1, 2023 9:00 PM', 'Apr 1, 2024 10:00 AM', 'Apr 1, 2024 10:00 PM', 'Apr 1, 2024 11:00 AM', 'Apr 1, 2024 11:00 PM', 'Apr 1, 2024 12:00 AM', 'Apr 1, 2024 12:00 PM', 'Apr 1, 2024 1:00 AM', 'Apr 1, 2024 1:00 PM', 'Apr 1, 2024 2:00 AM', 'Apr 1, 2024 2:00 PM', 'Apr 1, 2024 3:00 AM', 'Apr 1, 2024 3:00 PM', 'Apr 1, 2024 4:00 AM', 'Apr 1, 2024 4:00 PM', 'Apr 1, 2024 5:00 AM', 'Apr 1, 2024 5:00 PM', 'Apr 1, 2024 6:00 AM', 'Apr 1, 2024 6:00 PM', 'Apr 1, 2024 7:00 AM'

In [ ]:
temp_final["timestamp"] = temp_final["timestamp"].dt.tz_localize(None)

df_final = df.merge(temp_final, on="timestamp", how="inner")

print(df_final.shape)
print(df_final.head())


(26345, 7)
            timestamp   price       load wind_offshore wind_onshore solar   
0 2023-03-01 00:00:00  133.08  48,971.75        243.25     6,316.50  0.00  \
1 2023-03-01 01:00:00  125.46  47,543.75        243.75     6,137.25  0.00   
2 2023-03-01 02:00:00  124.96  47,152.00        229.00     5,835.25  0.00   
3 2023-03-01 03:00:00  127.66  47,447.25        198.75     5,595.00  0.00   
4 2023-03-01 04:00:00  130.58  49,152.25        154.00     5,600.00  0.00   

   temperature  
0      -2.8375  
1      -3.2750  
2      -3.7250  
3      -3.7125  
4      -3.6500  


In [74]:
df_final.head(2)

,timestamp,price,load,wind_offshore,wind_onshore,solar,temperature
0,2023-03-01 00:00:00,133.08,"48,971.75",243.25,"6,316.50",0.00,-2.8375
1,2023-03-01 01:00:00,125.46,"47,543.75",243.75,"6,137.25",0.00,-3.2750


In [76]:
df_final["timestamp"] = pd.to_datetime(df_final["timestamp"])

df_final["hour"]         = df_final["timestamp"].dt.hour
df_final["day_of_week"]  = df_final["timestamp"].dt.dayofweek
df_final["month"]        = df_final["timestamp"].dt.month
df_final["is_weekend"]   = df_final["timestamp"].dt.dayofweek >= 5

df_final.head(2)

,timestamp,price,load,wind_offshore,wind_onshore,solar,temperature,hour,day_of_week,month,is_weekend
0,2023-03-01 00:00:00,133.08,"48,971.75",243.25,"6,316.50",0.00,-2.8375,0,2,3,False
1,2023-03-01 01:00:00,125.46,"47,543.75",243.75,"6,137.25",0.00,-3.2750,1,2,3,False


In [78]:
df_final.shape

(26345, 11)